In [3]:
import pandas as pd

file_path = '../data/raw/cicids2017_cleaned.csv'
print(f"Loading dataset from {file_path}...\n")

# Load the dataset
df = pd.read_csv(file_path)

# Clean up column names (strips hidden spaces)
df.columns = df.columns.str.strip()

print(f"Dataset successfully loaded! Dimensions: {df.shape[0]} rows, {df.shape[1]} columns\n")

# Display the traffic class distribution using the correct column name
print("Traffic Class Distribution:")
print(df['Attack Type'].value_counts())

# Display the first 5 rows
display(df.head())

Loading dataset from ../data/raw/cicids2017_cleaned.csv...

Dataset successfully loaded! Dimensions: 2520751 rows, 53 columns

Traffic Class Distribution:
Attack Type
Normal Traffic    2095057
DoS                193745
DDoS               128014
Port Scanning       90694
Brute Force          9150
Web Attacks          2143
Bots                 1948
Name: count, dtype: int64


,Destination Port,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,...,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Max,Active Min,Idle Mean,Idle Max,Idle Min,Attack Type
0,22,1266342,41,2664,456,0,64.975610,109.864573,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
1,22,1319353,41,2664,456,0,64.975610,109.864573,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
2,22,160,1,0,0,0,0.000000,0.000000,0,0,...,243,0,32,0.0,0,0,0.0,0,0,Normal Traffic
3,22,1303488,41,2728,456,0,66.536585,110.129945,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
4,35396,77,1,0,0,0,0.000000,0.000000,0,0,...,290,0,32,0.0,0,0,0.0,0,0,Normal Traffic


In [4]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 1. Encode the Labels
# This converts text categories (e.g., 'Normal Traffic', 'DoS') into numbers (e.g., 0, 1)
print("Encoding target labels to numeric values...")
label_encoder = LabelEncoder()
df['Attack Type'] = label_encoder.fit_transform(df['Attack Type'])

# Show the mapping so you know which number represents which attack
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print("Label Mapping:", label_mapping)

# 2. Separate Features (X) and Target (y)
X = df.drop('Attack Type', axis=1) # All columns EXCEPT the attack type
y = df['Attack Type']              # ONLY the attack type column

# 3. Scale the Features
# This ensures all numerical features are on the same scale (mean=0, variance=1)
print("\nScaling feature data... (this might take a moment due to dataset size)")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert the scaled array back to a Pandas DataFrame for easier viewing
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

print("\nData Preprocessing Complete!")
print("Here is how your scaled features look now:")
display(X_scaled_df.head())

Encoding target labels to numeric values...
Label Mapping: {'Bots': np.int64(0), 'Brute Force': np.int64(1), 'DDoS': np.int64(2), 'DoS': np.int64(3), 'Normal Traffic': np.int64(4), 'Port Scanning': np.int64(5), 'Web Attacks': np.int64(6)}

Scaling feature data... (this might take a moment due to dataset size)

Data Preprocessing Complete!
Here is how your scaled features look now:


,Destination Port,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,...,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Max,Active Min,Idle Mean,Idle Max,Idle Min
0,-0.455939,-0.434952,0.038698,0.203389,0.297233,-0.315858,0.007536,0.109620,0.000527,-0.608918,...,1.502549,-0.222505,0.026695,0.002709,-0.133439,-0.15852,-0.10714,-0.37577,-0.381128,-0.361759
1,-0.455939,-0.433448,0.038698,0.203389,0.297233,-0.315858,0.007536,0.109620,0.000527,-0.608918,...,1.502549,-0.222505,0.026695,0.002709,-0.133439,-0.15852,-0.10714,-0.37577,-0.381128,-0.361759
2,-0.455939,-0.470891,-0.011655,-0.059957,-0.305693,-0.315858,-0.324733,-0.260441,-0.478338,-0.608918,...,-0.477852,-0.222505,-0.008893,0.002709,-0.133439,-0.15852,-0.10714,-0.37577,-0.381128,-0.361759
3,-0.455939,-0.433898,0.038698,0.209716,0.297233,-0.315858,0.015519,0.110514,0.000527,-0.608918,...,1.502549,-0.222505,0.026695,0.002709,-0.133439,-0.15852,-0.10714,-0.37577,-0.381128,-0.361759
4,1.404583,-0.470894,-0.011655,-0.059957,-0.305693,-0.315858,-0.324733,-0.260441,-0.478338,-0.608918,...,-0.481072,-0.217239,-0.008893,0.002709,-0.133439,-0.15852,-0.10714,-0.37577,-0.381128,-0.361759


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import time

print("1. Splitting data into Training (80%) and Testing (20%) sets...")
X_train, X_test, y_train, y_test = train_test_split(X_scaled_df, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training on {X_train.shape[0]} rows. Testing on {X_test.shape[0]} rows.")

print("\n2. Initializing the Random Forest AI Model...")
# n_estimators=50 uses 50 decision trees. It's a good balance of speed and accuracy for a first run.
# n_jobs=-1 tells the model to use all your CPU cores to train faster.
rf_model = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)

print("\n3. Training the AI model... (Go grab a coffee, this will take a few minutes!)")
start_time = time.time()
rf_model.fit(X_train, y_train)
end_time = time.time()
print(f"Training Complete! It took {round((end_time - start_time) / 60, 2)} minutes.")

print("\n4. Testing the model on unseen data...")
y_pred = rf_model.predict(X_test)

print("\n--- AI PERFORMANCE RESULTS ---")
print(f"Overall Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("Detailed Classification Report (Precision, Recall, F1-Score):")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

1. Splitting data into Training (80%) and Testing (20%) sets...
Training on 2016600 rows. Testing on 504151 rows.

2. Initializing the Random Forest AI Model...

3. Training the AI model... (Go grab a coffee, this will take a few minutes!)
Training Complete! It took 1.39 minutes.

4. Testing the model on unseen data...

--- AI PERFORMANCE RESULTS ---
Overall Accuracy: 99.90%

Detailed Classification Report (Precision, Recall, F1-Score):
                precision    recall  f1-score   support

          Bots       0.87      0.71      0.78       389
   Brute Force       1.00      1.00      1.00      1830
          DDoS       1.00      1.00      1.00     25603
           DoS       1.00      1.00      1.00     38749
Normal Traffic       1.00      1.00      1.00    419012
 Port Scanning       0.99      1.00      0.99     18139
   Web Attacks       0.99      0.97      0.98       429

      accuracy                           1.00    504151
     macro avg       0.98      0.95      0.96    5041

In [6]:
import joblib
import os

print("Exporting trained model and preprocessing tools...")

# Create the models directory if it doesn't already exist
os.makedirs('../models', exist_ok=True)

# 1. Save the trained Random Forest classifier
joblib.dump(rf_model, '../models/random_forest_model.pkl')

# 2. Save the fitted scaler (to normalize live incoming packets identically)
joblib.dump(scaler, '../models/scaler.pkl')

# 3. Save the label encoder (to map prediction numbers back to attack names)
joblib.dump(label_encoder, '../models/label_encoder.pkl')

# 4. Save the exact feature column names and order
joblib.dump(X_train.columns.tolist(), '../models/feature_names.pkl')

print("All artifacts successfully saved to '../models/'!")

Exporting trained model and preprocessing tools...
All artifacts successfully saved to '../models/'!
